In [ ]:
# Standard library imports
import os
import sys
import re
import logging
import warnings

# Third-party imports
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from xgboost import XGBRegressor

# Local imports
module_path = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..'))
if module_path not in sys.path:
    sys.path.append(module_path)
from paths import BASE_INPUT_PATH, BASE_OUTPUT_PATH

warnings.filterwarnings("ignore")

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)


def make_xgb_safe_columns(df):
    """
    XGBoost requires feature names to be strings and disallows [, ], <.
    """
    df = df.copy()
    seen = {}
    safe_cols = []

    for col in df.columns:
        c = str(col)
        c = re.sub(r"[\[\]<>]", "_", c)
        c = re.sub(r"[^0-9A-Za-z_]+", "_", c)
        c = re.sub(r"_+", "_", c).strip("_")
        if not c:
            c = "feature"

        if c in seen:
            seen[c] += 1
            c = f"{c}_{seen[c]}"
        else:
            seen[c] = 0

        safe_cols.append(c)

    df.columns = safe_cols
    return df


def build_features(df_target, df_exog=None, lags=(1, 2, 3, 7, 14, 28)):
    """
    Build supervised learning table for one-step-ahead forecasting.
    """
    feat = pd.DataFrame(index=df_target.index)
    feat["y"] = df_target.iloc[:, 0]

    for lag in lags:
        feat[f"lag_{lag}"] = feat["y"].shift(lag)

    feat["dow"] = feat.index.dayofweek
    feat["month"] = feat.index.month

    if df_exog is not None and not df_exog.empty:
        for col in df_exog.columns:
            feat[col] = df_exog[col]

    feat = feat.dropna()
    X = feat.drop(columns=["y"])
    y = feat["y"]
    return X, y


def xgboost_forecast(dataset, exo_dataset, product, product_sign, price_type, feature_nums, output_folder):
    logger.info(f"Processing {product_sign} {price_type} for {product} with features {feature_nums}...")

    dataset = dataset.loc[dataset["PRODUCT"] == product].copy()
    if dataset.empty:
        logger.warning(f"No data found for PRODUCT={product}")
        return None

    capacity_price_column = f"{product_sign}_GERMANY_{price_type}_CAPACITY_PRICE_[(EUR/MW)/h]"
    if capacity_price_column not in dataset.columns:
        logger.warning(f"Column '{capacity_price_column}' not found")
        return None

    y_raw = dataset[[capacity_price_column]].dropna().copy()
    df_init = y_raw.copy()

    pre_processing = ["0.95 capping", "log transformation"]

    if "0.95 capping" in pre_processing:
        q95 = y_raw.quantile(0.95).values[0]
        y_raw[y_raw > q95] = q95

    if "log transformation" in pre_processing:
        y_raw = np.log(y_raw).dropna()

    exo_product = None
    if exo_dataset is not None:
        exo_product = exo_dataset.loc[exo_dataset["PRODUCT"] == product].drop(columns=["PRODUCT"], errors="ignore")
        common_idx = y_raw.index.intersection(exo_product.index)
        y_raw = y_raw.loc[common_idx]
        exo_product = exo_product.loc[common_idx]

    X, y = build_features(y_raw, exo_product)

    # FIX: sanitize feature names for XGBoost
    X = make_xgb_safe_columns(X)

    n = len(X)
    train_end = int(n * 0.70)
    valid_end = int(n * 0.85)

    X_train, y_train = X.iloc[:train_end], y.iloc[:train_end]
    X_valid, y_valid = X.iloc[train_end:valid_end], y.iloc[train_end:valid_end]
    X_test, y_test = X.iloc[valid_end:], y.iloc[valid_end:]

    model = XGBRegressor(
        objective="reg:squarederror",
        n_estimators=800,
        learning_rate=0.03,
        max_depth=6,
        min_child_weight=3,
        subsample=0.9,
        colsample_bytree=0.9,
        reg_alpha=0.0,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=1
    )

    model.fit(
        X_train, y_train,
        eval_set=[(X_valid, y_valid)],
        verbose=False
    )

    valid_pred = model.predict(X_valid)
    test_pred = model.predict(X_test)

    if "log transformation" in pre_processing:
        valid_pred_inv = np.exp(valid_pred)
        test_pred_inv = np.exp(test_pred)
    else:
        valid_pred_inv = valid_pred
        test_pred_inv = test_pred

    valid_dates = X_valid.index
    test_dates = X_test.index
    valid_actual = df_init.loc[valid_dates].iloc[:, 0].values
    test_actual = df_init.loc[test_dates].iloc[:, 0].values

    valid_results_df = pd.DataFrame({
        "DATE": valid_dates,
        "ACTUAL_VALUE": valid_actual,
        "D+1": valid_pred_inv
    })

    test_results_df = pd.DataFrame({
        "DATE": test_dates,
        "ACTUAL_VALUE": test_actual,
        "D+1": test_pred_inv
    })

    results_df = pd.concat([valid_results_df, test_results_df], ignore_index=True)

    feature_nums_str = "_".join(map(str, feature_nums))
    results_df.to_csv(
        output_folder / f"xgboost_exog_{feature_nums_str}_model_results_{product}_{product_sign}_{price_type}.csv",
        index=False
    )

    overview_path = output_folder / f"xgboost_exog_{feature_nums_str}_models_overview.csv"
    if os.path.exists(overview_path):
        overview_df = pd.read_csv(overview_path, header=0)
    else:
        overview_df = pd.DataFrame()

    new_row = {
        "PRODUCT": product,
        "PRODUCT_SIGN": product_sign,
        "PRICE_TYPE": price_type,
        "EXOGENOUS_FACTORS": ", ".join(exo_product.columns) if exo_product is not None else "",
        "NUM_FEATURES": 0 if exo_product is None else len(exo_product.columns),
        "FEATURE_NUMBERS": feature_nums_str,
        "N_ESTIMATORS": model.get_params()["n_estimators"],
        "LEARNING_RATE": model.get_params()["learning_rate"],
        "MAX_DEPTH": model.get_params()["max_depth"]
    }

    overview_df = pd.concat([overview_df, pd.DataFrame([new_row])], ignore_index=True)
    overview_df = overview_df.sort_values(by=["PRODUCT", "PRODUCT_SIGN", "PRICE_TYPE"])
    overview_df.to_csv(overview_path, index=False)

    logger.info(f"Done: {product} {product_sign} {price_type}")
    return True


# Define products, signs, and price types
products = ["00_04", "04_08", "08_12", "12_16", "16_20", "20_24"]
product_signs = ["NEG", "POS"]
price_types = ["AVERAGE", "MARGINAL"]

# Read core dataset
dataset_path = BASE_INPUT_PATH / "processed_afrr_data.csv"
processed_afrr_data = pd.read_csv(dataset_path, parse_dates=["DATE"], index_col=["DATE"])

# Feature set selection
feature_nums = [1, 3]
exog_factors = {
    "feature_1": "FCR_GERMANY_SETTLEMENTCAPACITY_PRICE_[EUR/MW]",
    "feature_2": "CUMULATED_CAPACITY_MORE_EXPENSIVE_THAN_ELECTRICITY_PRICE_EXCESS_CAPACITY",
    "feature_3": "RES_SHARE",
    "feature_4": "EXESSIVE_AVAILABLE_CAPACITY_UNTIL_PRICE_LIMIT_9999_€/MWh",
    "feature_5": "CLEAN_SPREAD_OF_GAS"
}
selected_exog_features = [exog_factors[f"feature_{n}"] for n in feature_nums]

exo_dataset_path = BASE_INPUT_PATH / "selected_exogenous_factors_20210101_20250228.csv"
exogenous_factor_data = pd.read_csv(exo_dataset_path, parse_dates=["DATE"], index_col=["DATE"])
exogenous_factor_data = exogenous_factor_data[["PRODUCT"] + selected_exog_features].copy()

# Output folder
feature_nums_str = "_".join(map(str, feature_nums))
output_folder = BASE_OUTPUT_PATH / f"xgboost_exog_{feature_nums_str}_model_output"
output_folder.mkdir(parents=True, exist_ok=True)

# Align date index
common_dates = processed_afrr_data.index.intersection(exogenous_factor_data.index)
processed_afrr_data = processed_afrr_data.loc[common_dates]
exogenous_factor_data = exogenous_factor_data.loc[common_dates]

# Run all combinations
model_results = Parallel(n_jobs=1)(
    delayed(xgboost_forecast)(
        processed_afrr_data,
        exogenous_factor_data,
        product, product_sign, price_type,
        feature_nums,
        output_folder
    )
    for product in products
    for product_sign in product_signs
    for price_type in price_types
)